## 1. Setup & MNIST Download

Downloads the raw MNIST files, normalizes pixel values to `[0, 1]`, encodes labels as one-hot vectors, and writes two packed `f32` binary files (`data/mnist_train.bin`, `data/mnist_test.bin`) in the format expected by O.N.O (`x_size=784`, `y_size=10`, 794 floats per row).

In [21]:
import os, struct, gzip, shutil, urllib.request, subprocess, time
import numpy as np

NWORKERS = 3
NSERVERS = 2
BASE_WORKER_PORT = 50000
BASE_SERVER_PORT = 40000
WORKER_ADDRS = [f"worker-{i}:{BASE_WORKER_PORT + i}" for i in range(NWORKERS)]
SERVER_ADDRS = [f"server-{i}:{BASE_SERVER_PORT + i}" for i in range(NSERVERS)]

URLS = {
    "train_images": "https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz",
    "train_labels": "https://storage.googleapis.com/cvdf-datasets/mnist/train-labels-idx1-ubyte.gz",
    "test_images":  "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-images-idx3-ubyte.gz",
    "test_labels":  "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-labels-idx1-ubyte.gz",
}
RAW_DIR  = "data/mnist_raw"
TRAIN_BIN = "data/mnist_train.bin"
TEST_BIN  = "data/mnist_test.bin"
SAFETENSORS_PATH = "data/mnist_model.safetensors"
X_SIZE, Y_SIZE, ROW_SIZE = 784, 10, 794

def download(name, url):
    os.makedirs(RAW_DIR, exist_ok=True)
    gz, raw = os.path.join(RAW_DIR, name + ".gz"), os.path.join(RAW_DIR, name)
    if os.path.exists(raw):
        print(f"  already exists: {raw}"); return raw
    print(f"  downloading {name}...")
    urllib.request.urlretrieve(url, gz)
    with gzip.open(gz, "rb") as fi, open(raw, "wb") as fo:
        shutil.copyfileobj(fi, fo)
    os.remove(gz)
    return raw

def read_images(path):
    with open(path, "rb") as f:
        _, n, rows, cols = struct.unpack(">IIII", f.read(16))
        raw = f.read(n * rows * cols)
    ppi = rows * cols
    return [[b / 255.0 for b in raw[i*ppi:(i+1)*ppi]] for i in range(n)]

def read_labels(path):
    with open(path, "rb") as f:
        _, n = struct.unpack(">II", f.read(8))
        return list(f.read(n))

def write_bin(images, labels, out_path):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "wb") as f:
        for img, lbl in zip(images, labels):
            oh = [0.0] * Y_SIZE; oh[lbl] = 1.0
            f.write(struct.pack(f"{ROW_SIZE}f", *(img + oh)))
    print(f"  wrote {len(images)} samples → {out_path} ({os.path.getsize(out_path)/1024/1024:.1f} MB)")

for name, url in URLS.items():
    download(name, url)

if not os.path.exists(TRAIN_BIN):
    write_bin(read_images(os.path.join(RAW_DIR,"train_images")), read_labels(os.path.join(RAW_DIR,"train_labels")), TRAIN_BIN)
else:
    print(f"training set already exists: {TRAIN_BIN}")

if not os.path.exists(TEST_BIN):
    write_bin(read_images(os.path.join(RAW_DIR,"test_images")), read_labels(os.path.join(RAW_DIR,"test_labels")), TEST_BIN)
else:
    print(f"test set already exists: {TEST_BIN}")

  already exists: data/mnist_raw/train_images
  already exists: data/mnist_raw/train_labels
  already exists: data/mnist_raw/test_images
  already exists: data/mnist_raw/test_labels
training set already exists: data/mnist_train.bin
test set already exists: data/mnist_test.bin


## 2. Start Workers & Servers

Brings up `NSERVERS` parameter servers and `NWORKERS` workers as Docker containers. Each container logs via Docker. A short sleep is added after startup to ensure all entities are listening before the orchestrator connects.

In [ ]:
import getpass

DOCKER_DIR = os.path.abspath("../../docker")
COMPOSE_FILE = os.path.abspath("../../compose.yaml")

sudo_password = getpass.getpass("sudo password: ")

# Generate compose file
subprocess.run(
    ["python3", f"{DOCKER_DIR}/gen_compose.py"],
    env={
        **os.environ,
        "WORKERS": str(NWORKERS),
        "SERVERS": str(NSERVERS),
        "RELEASE": "true",
    },
    check=True,
)

# Fill /etc/hosts with worker/server hostname translations
subprocess.run(
    ["sudo", "-S", "-E", "python3", f"{DOCKER_DIR}/fill_hosts.py"],
    input=sudo_password + "\n",
    text=True,
    env={**os.environ, "WORKERS": str(NWORKERS), "SERVERS": str(NSERVERS)},
    check=True,
)

# Bring up containers
subprocess.run(
    ["docker", "compose", "-f", COMPOSE_FILE, "up", "--build", "-d", "--remove-orphans"],
    check=True,
)

time.sleep(3)
print("All entities running.")

#1 [internal] load local bake definitions
#1 reading from stdin 2.21kB done
#1 DONE 0.0s

#2 [worker-0 internal] load build definition from Dockerfile
#2 transferring dockerfile: 1.03kB done
#2 DONE 0.0s

#3 [server-0 internal] load build definition from Dockerfile
#3 transferring dockerfile: 982B done
#3 DONE 0.0s

#4 [server-1 internal] load metadata for docker.io/library/rust:1.92


## 3. Model Architecture

Defines the model architecture used for both distributed training (cell 3) and accuracy evaluation (cell 4). Each layer specifies an output size and an optional activation function. Supported activations: `"sigmoid"`, `None`.

In [ ]:
INPUT_SIZE = 784

ARCHITECTURE = [
    {"size": 128, "act": "sigmoid"},
    {"size": 64,  "act": "sigmoid"},
    {"size": 10,  "act": None},
]

## 4. Distributed Training with O.N.O

Builds a dense network from `ARCHITECTURE` with Kaiming initialization, then trains it distributedly using the Parameter Server algorithm (`NonBlockingSync`, `WildStore`, batch size 128, 300 epochs). Once complete, the trained weights are saved to `data/mnist_model.safetensors`.

In [ ]:
import orchestra
from orchestra import Sequential, orchestrate
from orchestra.arch import Dense
from orchestra.activations import Sigmoid
from orchestra.initialization import Kaiming
from orchestra.datasets import LocalDataset
from orchestra.optimizers import GradientDescent
from orchestra.sync import BarrierSync
from orchestra.store import BlockingStore
from orchestra.sync import NonBlockingSync
from orchestra.store import WildStore
import subprocess
import threading

ACT_MAP = {
    "sigmoid": Sigmoid(),
    None: None,
}

model = Sequential([
    Dense(layer["size"], Kaiming(), ACT_MAP[layer["act"]])
    for layer in ARCHITECTURE
])

training = orchestra.parameter_server(
    worker_addrs=WORKER_ADDRS,
    server_addrs=SERVER_ADDRS,
    dataset=LocalDataset(TRAIN_BIN, x_size=INPUT_SIZE, y_size=Y_SIZE),
    optimizer=GradientDescent(lr=0.1),
    sync=BarrierSync(),
    store=BlockingStore(),
    max_epochs=500,
    batch_size=256,
    offline_epochs=0,
    seed=42,
)

print("Starting training session...")
session = orchestrate(model, training)
trained = session.wait()
print(f"Training complete — {len(trained.weights())} parameters")

trained.save_safetensors(SAFETENSORS_PATH)
size = os.path.getsize(SAFETENSORS_PATH)
assert size > 0
print(f"Model saved to {SAFETENSORS_PATH} ({size} bytes)")

Starting training session...
  epoch 1/50 avg_loss=0.23746544
  epoch 1/50 avg_loss=0.23717451
  epoch 1/50 avg_loss=0.23728575
  epoch 2/50 avg_loss=0.36760417
  epoch 2/50 avg_loss=0.49773756
  epoch 2/50 avg_loss=0.62854475
  epoch 3/50 avg_loss=0.94058442
  epoch 3/50 avg_loss=1.25175416
  epoch 3/50 avg_loss=1.56397927
  epoch 4/50 avg_loss=1.19797170
  epoch 4/50 avg_loss=0.83051962
  epoch 4/50 avg_loss=0.46290788
  epoch 5/50 avg_loss=0.34996071
  epoch 5/50 avg_loss=0.23689246
  epoch 5/50 avg_loss=0.12316141
  epoch 6/50 avg_loss=0.11473515
  epoch 6/50 avg_loss=0.10643271
  epoch 6/50 avg_loss=0.09813759
  epoch 7/50 avg_loss=0.09593753
  epoch 7/50 avg_loss=0.09378634
  epoch 7/50 avg_loss=0.09161874
  epoch 8/50 avg_loss=0.09104814
  epoch 8/50 avg_loss=0.09045288
  epoch 8/50 avg_loss=0.08988988
  epoch 9/50 avg_loss=0.08973479
  epoch 9/50 avg_loss=0.08957604
  epoch 9/50 avg_loss=0.08940175
  epoch 10/50 avg_loss=0.08934403
  epoch 10/50 avg_loss=0.08929185
  epoch 10/5

## 5. Accuracy Evaluation with PyTorch

Loads the saved `.safetensors` file into an equivalent PyTorch model built dynamically from `ARCHITECTURE`. O.N.O stores weight matrices as `[input, output]` so they are transposed before loading into PyTorch's `Linear` layers (which expect `[output, input]`). The full 10,000-sample MNIST test set is then evaluated and top-1 accuracy is printed.

In [ ]:
import torch
import torch.nn as nn
from safetensors.torch import load_file

ACT_FN_MAP = {
    "sigmoid": torch.sigmoid,
    None: None,
}

class OnoNet(nn.Module):
    def __init__(self, input_size, architecture):
        super().__init__()
        sizes = [input_size] + [l["size"] for l in architecture]
        self.linears = nn.ModuleList([
            nn.Linear(sizes[i], sizes[i+1])
            for i in range(len(architecture))
        ])
        self.acts = [ACT_FN_MAP[l["act"]] for l in architecture]

    def forward(self, x):
        for linear, act in zip(self.linears, self.acts):
            x = linear(x)
            if act is not None:
                x = act(x)
        return x

state_dict = load_file(SAFETENSORS_PATH)
print("Loaded tensors:", list(state_dict.keys()))

net = OnoNet(INPUT_SIZE, ARCHITECTURE)

with torch.no_grad():
    for i, linear in enumerate(net.linears):
        linear.weight.copy_(state_dict[f"layer_{i}.weight"].T)
        linear.bias.copy_(state_dict[f"layer_{i}.bias"])

net.eval()

raw = np.fromfile(TEST_BIN, dtype=np.float32).reshape(-1, ROW_SIZE)
x_test = torch.tensor(raw[:, :INPUT_SIZE])
y_test = torch.tensor(raw[:, INPUT_SIZE:].argmax(axis=1), dtype=torch.long)

with torch.no_grad():
    preds = net(x_test).argmax(dim=1)
    acc = (preds == y_test).float().mean().item() * 100

print(f"\nTest accuracy: {acc:.2f}%")

Loaded tensors: ['layer_0.bias', 'layer_0.weight', 'layer_1.bias', 'layer_1.weight', 'layer_2.bias', 'layer_2.weight']

Test accuracy: 31.19%


## 6. Stop Workers & Servers

Terminates all worker and server subprocesses started in step 2 and closes their log file handles.

In [ ]:
print("Stopping all entities...")
subprocess.run(
    ["docker", "compose", "-f", COMPOSE_FILE, "down"],
    check=True,
)
print("Done.")

Stopping all entities...
Done.


 Container worker-2  Stopping
 Container worker-0  Stopping
 Container worker-1  Stopping
 Container worker-1  Stopped
 Container worker-1  Removing
 Container worker-0  Stopped
 Container worker-0  Removing
 Container worker-2  Stopped
 Container worker-2  Removing
 Container worker-0  Removed
 Container worker-2  Removed
 Container worker-1  Removed
 Container server-0  Stopping
 Container server-1  Stopping
 Container server-1  Stopped
 Container server-1  Removing
 Container server-0  Stopped
 Container server-0  Removing
 Container server-1  Removed
 Container server-0  Removed
 Network distributed-training_training-network  Removing
 Network distributed-training_training-network  Removed
